# 🧠 global spectral geometry + dominant-mode spatial distribution — ML INDIVIDUAL COMPLETO (versión unificada y mejorada)

---

## Qué mejora respecto a las versiones anteriores

| Mejora | Antes | Ahora |
|---|---|---|
| `clean_feature_block` | Solo dominant-mode spatial distribution | **Ambos** — filtra NaN, varianza, tamaño mínimo |
| Threshold OOF tuning | Solo dominant-mode spatial distribution | **Ambos** — grid 0.30→0.70 ajustado en train |
| `HGB` (HistGradientBoosting) | Solo dominant-mode spatial distribution | **Ambos** |
| `RFF_LinearSVC` | Solo dominant-mode spatial distribution | **Ambos** |
| `SelectKBest` (SKB) | Solo dominant-mode spatial distribution | **Ambos** |
| `delta_pooled` scope | Solo dominant-mode spatial distribution | **Ambos** |
| `MIN_SUBJECTS_PER_BLOCK` | Solo dominant-mode spatial distribution | **Ambos** |
| RFF gammas | global spectral geometry: [0.5,1,2] / dominant-mode spatial distribution: [0.25,0.5,1,2] | **Ambos: [0.25, 0.5, 1.0, 2.0]** |
| RFF components | global spectral geometry: 300 / dominant-mode spatial distribution: 400 | **Ambos: 400** |
| `col_as_series` robusto | Solo dominant-mode spatial distribution | **Ambos** |

## Estructura

| Sección | Contenido | Cuándo ejecutar |
|---|---|---|
| 0 | Imports, CONFIG y rutas | ⚠️ Siempre primero |
| 1 | Feature sets global spectral geometry y dominant-mode spatial distribution | ⚠️ Siempre primero |
| 2 | Helpers compartidos (CV, métricas, modelos, limpieza) | ⚠️ Siempre primero |
| 3 | Carga de datos | ⚠️ Siempre primero |
| **A** | **global spectral geometry ML** — bywin / pooled / delta_bywin / delta_pooled | 🔵 Independiente |
| **B** | **dominant-mode spatial distribution ML** — bywin / pooled / delta_bywin / delta_pooled | 🟣 Independiente |
| **C** | **Visualización** — comparación global spectral geometry vs dominant-mode spatial distribution | 📊 Lee CSVs de A y B |

This notebook is part of the public analysis repository associated with the EEG eigenmode study. It uses precomputed feature tables generated by the preprocessing and feature extraction scripts. Raw EEG recordings and participant-level data are not included because the study involves minors and is subject to ethical and privacy restrictions.

---
# SECCIÓN 0 — Imports y CONFIG
> ⚠️ Ejecutar siempre primero

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.kernel_approximation import RBFSampler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.feature_selection import SelectKBest, mutual_info_classif

try:
    from sklearn.model_selection import StratifiedGroupKFold
    HAS_SGK = True
except Exception:
    HAS_SGK = False

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", font_scale=1.05)

# ── RUTAS ──────────────────────────────────────────────────────────────────────
BASE   = Path(r"")
GSG_DIR = BASE / "results/features/global_spectral_geometry"
DMSD_DIR = BASE / "results/features/dominant_mode_spatial_distribution"

GSG_BYWIN        = GSG_DIR / "01_features" / "GSG_features_by_win.csv"
GSG_POOLED       = GSG_DIR / "01_features" / "GSG_features_pooled.csv"
GSG_DELTA_BYWIN  = GSG_DIR / "07_delta_basal_to_pvt" / "GSG_delta_table_by_win.csv"
# global spectral geometry delta pooled: si existe, usarlo; si no, se salta silenciosamente
GSG_DELTA_POOLED = GSG_DIR / "07_delta_basal_to_pvt" / "GSG_delta_table_pooled.csv"

DMSD_BYWIN        = DMSD_DIR / "01_features" / "DMSD_features_by_win.csv"
DMSD_POOLED       = DMSD_DIR / "01_features" / "DMSD_features_pooled.csv"
DMSD_DELTA_BYWIN  = DMSD_DIR / "01_features" / "DMSD_delta_by_win.csv"
DMSD_DELTA_POOLED = DMSD_DIR / "01_features" / "DMSD_delta_pooled.csv"

OUT_GSG = GSG_DIR / "08_ml_feature_sets_v1"; OUT_GSG.mkdir(parents=True, exist_ok=True)
OUT_DMSD = DMSD_DIR / "03_ml_feature_sets_v1"; OUT_DMSD.mkdir(parents=True, exist_ok=True)



# ── SALIDAS ORDENADAS ─────────────────────────────────────────────────────────
OUT_GSG_RAW      = OUT_GSG / "01_raw_tables";        OUT_GSG_RAW.mkdir(parents=True, exist_ok=True)
OUT_GSG_SUMMARY  = OUT_GSG / "02_summaries";         OUT_GSG_SUMMARY.mkdir(parents=True, exist_ok=True)
OUT_GSG_RANK     = OUT_GSG / "03_rankings";          OUT_GSG_RANK.mkdir(parents=True, exist_ok=True)

OUT_DMSD_RAW      = OUT_DMSD / "01_raw_tables";        OUT_DMSD_RAW.mkdir(parents=True, exist_ok=True)
OUT_DMSD_SUMMARY  = OUT_DMSD / "02_summaries";         OUT_DMSD_SUMMARY.mkdir(parents=True, exist_ok=True)
OUT_DMSD_RANK     = OUT_DMSD / "03_rankings";          OUT_DMSD_RANK.mkdir(parents=True, exist_ok=True)

OUT_GLOBAL      = BASE / "EIGENMODE_combined eigenmode representations_GLOBAL_ML_V1"; OUT_GLOBAL.mkdir(parents=True, exist_ok=True)
OUT_GLOBAL_RANK = OUT_GLOBAL / "01_rankings";      OUT_GLOBAL_RANK.mkdir(parents=True, exist_ok=True)

def rank_cols_for_export(df):
    cols = []
    for c in ["subj_bal_acc_mean", "bal_acc_mean", "subj_f1_macro_mean", "subj_auc_mean", "f1_macro_mean", "auc_mean"]:
        if c in df.columns:
            cols.append(c)
    return cols

def save_ranked_csv(df, path):
    if df is None or df.empty:
        return
    rank_cols = rank_cols_for_export(df)
    if rank_cols:
        df = df.sort_values(rank_cols, ascending=[False] * len(rank_cols))
    df.to_csv(path, index=False, float_format=CSV_FLOAT_FMT)

# ── HIPERPARÁMETROS ────────────────────────────────────────────────────────────
CSV_FLOAT_FMT            = "%.8e"
SEED                     = 123
N_SPLITS_DESIRED         = 5
MIN_ROWS_PER_BLOCK       = 20
MIN_SUBJECTS_PER_BLOCK   = 12   # ← ahora también en global spectral geometry
MIN_GROUPS_PER_CLASS     = 2
MAX_NAN_FRAC_PER_FEATURE = 0.40  # ← ahora también en global spectral geometry
MIN_VAR_PER_FEATURE      = 1e-12 # ← ahora también en global spectral geometry
MIN_FEATURES_AFTER_CLEAN = 4     # ← ahora también en global spectral geometry
RFF_GAMMAS               = [0.25, 0.5, 1.0, 2.0]  # ← unificados
RFF_COMPONENTS           = 400                      # ← unificados
SELECT_K_OPTIONS         = [6, 10, 16]              # ← ahora también en global spectral geometry
THRESH_GRID              = np.linspace(0.30, 0.70, 17)  # ← ahora también en global spectral geometry

print("✅ CONFIG OK")
print(f"   OUT global spectral geometry → {OUT_GSG}")
print(f"   OUT dominant-mode spatial distribution → {OUT_DMSD}")

---
# SECCIÓN 1 — Feature sets
> ⚠️ Ejecutar siempre primero

In [3]:
# ══════════════════════════════════════════════════════
# global spectral geometry
# ══════════════════════════════════════════════════════
GSG_RADIAL = [
    "all__rad_median","all__rad_p95","all__rad_std",
    "all__rad_signed_median","all__rad_prop_in","all__rad_prop_out",
    "all__rad_abs_mean","all__rad_abs_std",
    "all__rad_signed_mean","all__rad_signed_std",
    "all__dist_uc_abs_mean","all__dist_uc_abs_std",
    "osc__rad_median","osc__rad_p95","osc__rad_std",
    "osc__rad_signed_median","osc__rad_prop_in","osc__rad_prop_out",
    "osc__rad_abs_mean","osc__rad_abs_std",
    "osc__rad_signed_mean","osc__rad_signed_std",
    "osc__dist_uc_abs_mean","osc__dist_uc_abs_std",
    "all__dist_p95",
    "all__dist_prop_gt_0.02","all__dist_prop_gt_0.05","all__dist_prop_gt_0.10",
    "osc__dist_p95",
    "osc__dist_prop_gt_0.02","osc__dist_prop_gt_0.05","osc__dist_prop_gt_0.10",
    "all__dist_prop_gt_aacc_p95","osc__dist_prop_gt_aacc_p95",
]

GSG_ANGLE = [
    "osc_frac","osc__angle_p95_deg",
    "osc__angle_prop_gt_35deg","osc__angle_prop_gt_45deg",
    "osc__ang_abs_mean_deg","osc__ang_abs_std_deg",
    "osc__imag_abs_mean","osc__imag_abs_std",
    "osc__angle_prop_gt_aacc_p95","osc__score_prop_gt_aacc_p95",
]

GSG_RING = [
    "osc__ring0.02__angle_p95_deg","osc__ring0.02__angle_prop_gt_35deg","osc__ring0.02__angle_prop_gt_45deg",
    "osc__ring0.05__angle_p95_deg","osc__ring0.05__angle_prop_gt_35deg","osc__ring0.05__angle_prop_gt_45deg",
]

GSG_PRIORITY = [
    "all__dist_prop_gt_0.02","all__dist_p95","osc_frac",
    "osc__rad_signed_median","osc__rad_prop_out","osc__rad_prop_in",
    "osc__dist_uc_abs_mean","osc__rad_signed_mean","osc__imag_abs_mean",
    "osc__ang_abs_std_deg",
    "osc__ring0.02__angle_prop_gt_35deg","osc__ring0.05__angle_prop_gt_35deg",
    "osc__angle_p95_deg","osc__angle_prop_gt_35deg","osc__dist_prop_gt_0.05",
    "osc__angle_prop_gt_aacc_p95","all__dist_prop_gt_aacc_p95",
]

GSG_DELTA_FOCUS = [
    "delta__all__rad_signed_median","delta__all__rad_prop_out",
    "delta__all__dist_uc_abs_mean","delta__osc_frac",
    "delta__osc__imag_abs_mean",
    "delta__osc__ring0.02__angle_prop_gt_35deg","delta__osc__ring0.05__angle_prop_gt_35deg",
    "delta__all__dist_prop_gt_0.02","delta__all__dist_p95",
    "delta__osc__rad_prop_out","delta__osc__rad_signed_median","delta__osc__dist_uc_abs_mean",
]

# ══════════════════════════════════════════════════════
# dominant-mode spatial distribution
# ══════════════════════════════════════════════════════
DMSD_PRIMARY = [
    "s_core","s_temporal","s_central","s_frontal","s_parietal",
    "entropy_norm","hhi","n_eff","s_core_star",
]
DMSD_DISTRIBUTION = [
    "max_p","top4_mass","gini","kurtosis_p",
    "core_to_rest","dom_max_region","dom_gap_core_vs_best_other",
]
DMSD_REGION_LOGITS  = ["logit_core","logit_frontal","logit_central","logit_temporal","logit_parietal"]
DMSD_REGION_STARS   = ["s_frontal_star","s_central_star","s_temporal_star","s_parietal_star"]
DMSD_DISJOINT       = ["sD_core","sD_frontal","sD_central","sD_parietal_nocore","sD_temporal_nocore","sD_rest"]
DMSD_CONCENTRATION  = ["entropy_norm","hhi","n_eff","max_p","top4_mass","gini","kurtosis_p"]
DMSD_CORE_VS_OTHERS = [
    "s_core","s_core_star","core_to_rest",
    "s_temporal","s_central","s_frontal","s_parietal",
    "dom_max_region","dom_gap_core_vs_best_other",
]
DMSD_INTERPRETABLE_COMPACT = [
    "s_core","s_core_star",
    "s_temporal","s_central","s_frontal","s_parietal",
    "entropy_norm","hhi","n_eff",
    "core_to_rest","dom_gap_core_vs_best_other",
    "sD_core","sD_central","sD_frontal","sD_parietal_nocore","sD_rest",
]
DMSD_PRIORITY_STAT  = ["s_core","s_core_star","logit_core","sD_core","core_to_rest","dom_gap_core_vs_best_other","s_temporal","logit_temporal"]
DMSD_PRIORITY_NO_OCCIPITAL = ["s_core","s_core_star","s_temporal","logit_core","logit_temporal","sD_core","sD_rest","core_to_rest","dom_gap_core_vs_best_other"]
DMSD_PRIORITY_DELTA = ["delta_s_core","delta_sD_core","delta_logit_core","delta_core_to_rest","delta_s_core_star","delta_rest_shift","delta_s_temporal","delta_s_temporal_star"]
DMSD_DELTA_REORG    = ["delta_reorg_L1","delta_reorg_L2","delta_core_vs_central_shift","delta_rest_shift"]
DMSD_DELTA_DISJOINT = ["delta_sD_core","delta_sD_frontal","delta_sD_central","delta_sD_parietal_nocore","delta_sD_temporal_nocore","delta_sD_rest"]
DMSD_DELTA_COMPACT  = [
    "delta_s_core","delta_s_core_star",
    "delta_s_temporal","delta_s_central","delta_s_frontal","delta_s_parietal",
    "delta_entropy_norm","delta_hhi","delta_n_eff",
    "delta_core_to_rest","delta_dom_gap_core_vs_best_other",
    "delta_reorg_L1","delta_reorg_L2","delta_core_vs_central_shift","delta_rest_shift",
]

print("✅ Feature sets OK")

---
# SECCIÓN 2 — Helpers compartidos (unificados)
> ⚠️ Ejecutar siempre primero

In [4]:
# ── Utilidades generales ───────────────────────────────────────────────────────
def dedupe_keep_order(seq):
    seen, out = set(), []
    for x in seq:
        if x not in seen: out.append(x); seen.add(x)
    return out

def col_as_series(df, c):
    """Devuelve siempre una Serie, aunque la columna esté duplicada."""
    x = df.loc[:, c]
    return x.iloc[:, 0] if isinstance(x, pd.DataFrame) else x

def available_features(df, candidates):
    return dedupe_keep_order([c for c in candidates if c in df.columns])

def all_numeric_feature_cols(df, meta_cols):
    out = []
    for c in df.columns:
        if c in meta_cols: continue
        if pd.api.types.is_numeric_dtype(col_as_series(df, c)): out.append(c)
    return dedupe_keep_order(out)

# ── CV ─────────────────────────────────────────────────────────────────────────
def make_group_cv(y, groups, n_splits_desired=5):
    y = np.asarray(y).astype(int)
    groups = np.asarray(groups).astype(str)
    n_splits = int(min(max(2, n_splits_desired), len(np.unique(groups))))
    if n_splits < 2: return None
    if HAS_SGK:
        tmp = pd.DataFrame({"g": groups, "y": y}).drop_duplicates("g")
        if int((tmp["y"]==0).sum()) >= n_splits and int((tmp["y"]==1).sum()) >= n_splits:
            return StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    return GroupKFold(n_splits=n_splits)

# ── Métricas ───────────────────────────────────────────────────────────────────
def safe_auc(y_true, y_score):
    try: return roc_auc_score(y_true, y_score)
    except: return np.nan

def to_prob_like(score):
    score = np.asarray(score, float)
    if score.size == 0: return score
    if np.nanmin(score) < 0 or np.nanmax(score) > 1:
        score = 1.0 / (1.0 + np.exp(-score))
    return score

def choose_best_threshold(y_true, y_score):
    """Busca el threshold óptimo (F1-macro) sobre un grid en [0.30, 0.70]."""
    best_thr, best_f1 = 0.5, -np.inf
    for thr in THRESH_GRID:
        yhat = (np.asarray(y_score, float) >= thr).astype(int)
        try: f1m = f1_score(y_true, yhat, average="macro")
        except: f1m = np.nan
        if np.isfinite(f1m) and f1m > best_f1: best_f1, best_thr = f1m, float(thr)
    return best_thr, best_f1

def aggregate_subject_scores(scores, ids, thr=0.5):
    agg = (pd.DataFrame({"id": ids, "score": scores})
             .groupby("id", as_index=False)["score"].mean())
    agg["yhat"] = (agg["score"] >= thr).astype(int)
    return agg

# ── Fit / predict / OOF ────────────────────────────────────────────────────────
def fit_predict_scores(model, Xtr, ytr, Xte):
    model.fit(Xtr, ytr)
    yhat = model.predict(Xte)
    score = None
    if hasattr(model, "predict_proba"):
        try: score = model.predict_proba(Xte)[:, 1]
        except: pass
    if score is None and hasattr(model, "decision_function"):
        try: score = model.decision_function(Xte)
        except: pass
    return yhat, score

def get_oof_scores(model, X, y, groups):
    """OOF scores para threshold tuning. Devuelve None si falla."""
    cv_in = make_group_cv(y, groups, n_splits_desired=min(4, N_SPLITS_DESIRED))
    if cv_in is None: return None
    oof = np.full(len(X), np.nan, dtype=float)
    for tr, va in cv_in.split(X, y, groups):
        mdl = clone(model)
        _, sv = fit_predict_scores(mdl, X.iloc[tr], y[tr], X.iloc[va])
        if sv is None: return None
        oof[va] = to_prob_like(sv)
    return None if np.any(~np.isfinite(oof)) else oof

# ── Limpieza de features (ahora compartida por global spectral geometry y dominant-mode spatial distribution) ───────────────────────
def clean_feature_block(df_block, feature_cols):
    """
    Elimina features con demasiados NaN, muy poca varianza o muy pocos finitos.
    Devuelve (cols_limpias, cols_eliminadas).
    """
    keep, dropped = [], []
    feature_cols = dedupe_keep_order([c for c in feature_cols if c in df_block.columns])
    for c in feature_cols:
        x  = pd.to_numeric(col_as_series(df_block, c), errors="coerce").values.astype(float)
        xf = x[np.isfinite(x)]
        nan_frac = float(np.mean(~np.isfinite(x)))
        var = float(np.var(xf)) if xf.size > 0 else np.nan
        if nan_frac > MAX_NAN_FRAC_PER_FEATURE:           dropped.append((c, "nan_frac",  nan_frac)); continue
        if xf.size < 8:                                    dropped.append((c, "too_few",   xf.size));  continue
        if np.isfinite(var) and var <= MIN_VAR_PER_FEATURE: dropped.append((c, "low_var",   var));      continue
        keep.append(c)
    return dedupe_keep_order(keep), dropped

def build_X_numeric(df_block, feature_cols):
    X = pd.DataFrame(index=df_block.index)
    for c in feature_cols:
        X[c] = pd.to_numeric(col_as_series(df_block, c), errors="coerce")
    return X

# ── Modelos (ahora idénticos para global spectral geometry y dominant-mode spatial distribution) ────────────────────────────────────
def make_models(n_features):
    """
    Devuelve el dict completo de modelos para un bloque con n_features limpias.
    SKB solo se añade si n_features > k.
    """
    m = {}

    # Base
    m["LR"] = Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc",  StandardScaler()),
        ("clf", LogisticRegression(solver="liblinear", class_weight="balanced",
                                   max_iter=8000, random_state=SEED))
    ])
    m["LinearSVC_cal"] = Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc",  StandardScaler()),
        ("clf", CalibratedClassifierCV(
            estimator=LinearSVC(class_weight="balanced", random_state=SEED),
            method="sigmoid", cv=3))
    ])
    m["SVM_RBF"] = Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc",  StandardScaler()),
        ("clf", SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced",
                    probability=True, random_state=SEED))
    ])
    m["ExtraTrees"] = Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("clf", ExtraTreesClassifier(n_estimators=1200, random_state=SEED,
                                     class_weight="balanced", n_jobs=-1))
    ])
    m["HGB"] = Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("clf", HistGradientBoostingClassifier(random_state=SEED))
    ])

    # RFF variants
    for g in RFF_GAMMAS:
        m[f"RFF_LR_g{g}"] = Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc",  StandardScaler()),
            ("rff", RBFSampler(gamma=g, n_components=RFF_COMPONENTS, random_state=SEED)),
            ("clf", LogisticRegression(solver="liblinear", class_weight="balanced",
                                       max_iter=8000, random_state=SEED))
        ])
        m[f"RFF_LinearSVC_g{g}"] = Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc",  StandardScaler()),
            ("rff", RBFSampler(gamma=g, n_components=RFF_COMPONENTS, random_state=SEED)),
            ("clf", CalibratedClassifierCV(
                estimator=LinearSVC(class_weight="balanced", random_state=SEED),
                method="sigmoid", cv=3))
        ])

    # SKB variants (solo si hay features suficientes)
    for k in [kk for kk in SELECT_K_OPTIONS if kk < n_features]:
        m[f"SKB{k}_LR"] = Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc",  StandardScaler()),
            ("skb", SelectKBest(score_func=mutual_info_classif, k=k)),
            ("clf", LogisticRegression(solver="liblinear", class_weight="balanced",
                                       max_iter=8000, random_state=SEED))
        ])
        m[f"SKB{k}_LinearSVC"] = Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc",  StandardScaler()),
            ("skb", SelectKBest(score_func=mutual_info_classif, k=k)),
            ("clf", CalibratedClassifierCV(
                estimator=LinearSVC(class_weight="balanced", random_state=SEED),
                method="sigmoid", cv=3))
        ])
        for g in RFF_GAMMAS:
            m[f"SKB{k}_RFF_LR_g{g}"] = Pipeline([
                ("imp", SimpleImputer(strategy="median")),
                ("sc",  StandardScaler()),
                ("skb", SelectKBest(score_func=mutual_info_classif, k=k)),
                ("rff", RBFSampler(gamma=g, n_components=RFF_COMPONENTS, random_state=SEED)),
                ("clf", LogisticRegression(solver="liblinear", class_weight="balanced",
                                           max_iter=8000, random_state=SEED))
            ])
    return m

# ── Core evaluate (compartido global spectral geometry y dominant-mode spatial distribution) ────────────────────────────────────────
def evaluate_block(df_block, feature_cols, tag, fs_name):
    """
    Evalúa un bloque con todos los modelos.
    Incluye: limpieza, threshold OOF tuning, métricas por ventana y por sujeto.
    """
    feature_cols = dedupe_keep_order([c for c in feature_cols if c in df_block.columns])
    if len(feature_cols) < 4: return pd.DataFrame()

    X      = build_X_numeric(df_block, feature_cols)
    y      = df_block["y"].astype(int).values
    groups = df_block["id"].astype(str).values

    tmp = pd.DataFrame({"g": groups, "y": y}).drop_duplicates("g")
    if tmp["y"].nunique() < 2: return pd.DataFrame()
    if int((tmp["y"]==0).sum()) < MIN_GROUPS_PER_CLASS or \
       int((tmp["y"]==1).sum()) < MIN_GROUPS_PER_CLASS: return pd.DataFrame()

    cv = make_group_cv(y, groups, n_splits_desired=N_SPLITS_DESIRED)
    if cv is None: return pd.DataFrame()

    # Limpieza de features (ahora en ambos global spectral geometry y dominant-mode spatial distribution)
    cleaned_cols, dropped_cols = clean_feature_block(df_block, feature_cols)
    if len(cleaned_cols) < MIN_FEATURES_AFTER_CLEAN: return pd.DataFrame()
    X = X[cleaned_cols].copy()

    rows = []
    for model_name, model in make_models(len(cleaned_cols)).items():
        bal_list, f1_list, auc_list = [], [], []
        sbal_list, sf1_list, sauc_list, subj_n, thr_list = [], [], [], [], []

        for tr, te in cv.split(X, y, groups):
            Xtr, Xte = X.iloc[tr].copy(), X.iloc[te].copy()
            ytr, yte = y[tr], y[te]
            gtr, gte = groups[tr], groups[te]

            # Threshold tuning con OOF sobre el train (sin fuga)
            oof_tr = get_oof_scores(model, Xtr, ytr, gtr)
            thr    = choose_best_threshold(ytr, oof_tr)[0] if oof_tr is not None else 0.5
            thr_list.append(thr)

            _, score_te = fit_predict_scores(clone(model), Xtr, ytr, Xte)

            if score_te is None:
                mdl = clone(model); mdl.fit(Xtr, ytr)
                yhat_te = mdl.predict(Xte)
                bal_list.append(balanced_accuracy_score(yte, yhat_te))
                f1_list.append(f1_score(yte, yhat_te, average="macro"))
                auc_list.append(np.nan)
                continue

            score_te = to_prob_like(score_te)
            yhat_te  = (score_te >= thr).astype(int)
            bal_list.append(balanced_accuracy_score(yte, yhat_te))
            f1_list.append(f1_score(yte, yhat_te, average="macro"))
            auc_list.append(safe_auc(yte, score_te))

            agg   = aggregate_subject_scores(score_te, gte, thr=thr)
            y_sub = pd.DataFrame({"id":gte,"y":yte}).groupby("id",as_index=False)["y"].first()
            m     = agg.merge(y_sub, on="id", how="inner")
            if len(m) >= 4:
                sbal_list.append(balanced_accuracy_score(m["y"].values, m["yhat"].values))
                sf1_list.append(f1_score(m["y"].values, m["yhat"].values, average="macro"))
                sauc_list.append(safe_auc(m["y"].values, m["score"].values))
                subj_n.append(len(m))

        rows.append({
            "tag":tag, "feature_set":fs_name, "model":model_name,
            "n_features":         len(cleaned_cols),
            "n_features_dropped": len(dropped_cols),
            "n_samples":          len(df_block),
            "n_subjects":         int(df_block["id"].nunique()),
            "n_aacc_subjects":    int(df_block[df_block["y"]==1]["id"].nunique()),
            "n_ctrl_subjects":    int(df_block[df_block["y"]==0]["id"].nunique()),
            "bal_acc_mean":       float(np.nanmean(bal_list)),
            "bal_acc_std":        float(np.nanstd(bal_list, ddof=1)) if len(bal_list)>1 else np.nan,
            "f1_macro_mean":      float(np.nanmean(f1_list)),
            "f1_macro_std":       float(np.nanstd(f1_list, ddof=1)) if len(f1_list)>1 else np.nan,
            "auc_mean":           float(np.nanmean(auc_list)),
            "auc_std":            float(np.nanstd(auc_list, ddof=1)) if len(auc_list)>1 else np.nan,
            "subj_bal_acc_mean":  float(np.nanmean(sbal_list)) if sbal_list else np.nan,
            "subj_bal_acc_std":   float(np.nanstd(sbal_list, ddof=1)) if len(sbal_list)>1 else np.nan,
            "subj_f1_macro_mean": float(np.nanmean(sf1_list)) if sf1_list else np.nan,
            "subj_f1_macro_std":  float(np.nanstd(sf1_list, ddof=1)) if len(sf1_list)>1 else np.nan,
            "subj_auc_mean":      float(np.nanmean(sauc_list)) if sauc_list else np.nan,
            "subj_auc_std":       float(np.nanstd(sauc_list, ddof=1)) if len(sauc_list)>1 else np.nan,
            "subj_n_test_mean":   float(np.nanmean(subj_n)) if subj_n else np.nan,
            "threshold_mean":     float(np.nanmean(thr_list)) if thr_list else np.nan,
            "threshold_std":      float(np.nanstd(thr_list, ddof=1)) if len(thr_list)>1 else np.nan,
        })
    return pd.DataFrame(rows)

# ── Summary helper genérico ────────────────────────────────────────────────────
def save_summary(df_res, out_dir, prefix):
    if df_res is None or df_res.empty: return
    sc = ["subj_f1_macro_mean","subj_bal_acc_mean","subj_auc_mean","f1_macro_mean","auc_mean"]
    sc_asc = [False]*len(sc)
    (df_res.sort_values(["tag"]+sc, ascending=[True]+sc_asc)
           .groupby("tag", as_index=False).head(1)
           .to_csv(out_dir/f"{prefix}_best_by_tag.csv", index=False, float_format=CSV_FLOAT_FMT))
    (df_res.sort_values(["tag","feature_set"]+sc, ascending=[True,True]+sc_asc)
           .groupby(["tag","feature_set"], as_index=False).head(1)
           .to_csv(out_dir/f"{prefix}_best_by_tag_and_fs.csv", index=False, float_format=CSV_FLOAT_FMT))
    (df_res.groupby("model", as_index=False)[sc].mean(numeric_only=True)
           .sort_values(sc[:2], ascending=[False,False])
           .to_csv(out_dir/f"{prefix}_mean_by_model.csv", index=False, float_format=CSV_FLOAT_FMT))
    (df_res.groupby("feature_set", as_index=False)[sc].mean(numeric_only=True)
           .sort_values(sc[:2], ascending=[False,False])
           .to_csv(out_dir/f"{prefix}_mean_by_feature_set.csv", index=False, float_format=CSV_FLOAT_FMT))

print("✅ Helpers OK")

---
# SECCIÓN 3 — Carga de datos
> ⚠️ Ejecutar siempre primero

In [ ]:
def load_all():
    paths = {
        "GSG_bywin":        GSG_BYWIN,
        "GSG_pooled":       GSG_POOLED,
        "GSG_delta_bywin":  GSG_DELTA_BYWIN,
        "GSG_delta_pooled": GSG_DELTA_POOLED,   # opcional
        "DMSD_bywin":        DMSD_BYWIN,
        "DMSD_pooled":       DMSD_POOLED,
        "DMSD_delta_bywin":  DMSD_DELTA_BYWIN,
        "DMSD_delta_pooled": DMSD_DELTA_POOLED,
    }
    data = {}
    print("Datos cargados:")
    for k, p in paths.items():
        if p.exists():
            data[k] = pd.read_csv(p)
            print(f"  {k:22s} → {data[k].shape[0]:5d} filas | {data[k]['id'].nunique():3d} sujetos")
        else:
            data[k] = pd.DataFrame()
            print(f"  {k:22s} → ⚠️  no encontrado ({p.name})")
    return data

DATA = load_all()

---
# 🔵 BLOQUE A — global spectral geometry ML

**Novedades respecto a la versión anterior:**
- `clean_feature_block` activo
- Threshold OOF tuning (grid 0.30→0.70)
- Modelos nuevos: `HGB`, `RFF_LinearSVC`, `SKB` variants
- Nuevo scope: `delta_pooled` (si existe el CSV)
- `MIN_SUBJECTS_PER_BLOCK = 12` en todos los scopes
- RFF gammas unificados: [0.25, 0.5, 1.0, 2.0] con 400 componentes

In [ ]:
# ── Feature set builders global spectral geometry ────────────────────────────────────────────────────
_GSG_META_WIN   = {"id","group","y","cond","win_sec","tercile","W_used","N_eigs","n_points","filter_imag_pos","osc_tau"}
_GSG_META_POOL  = {"id","group","y","cond","tercile","filter_imag_pos","osc_tau"}
_GSG_META_DELTA = {"id","group","y","win_sec","tercile"}

def _GSG_static_sets(df, meta):
    af = all_numeric_feature_cols(df, meta)
    sets = {
        "priority":        available_features(df, GSG_PRIORITY),
        "radial":          available_features(df, GSG_RADIAL),
        "angle":           available_features(df, GSG_ANGLE),
        "ring":            available_features(df, GSG_RING),
        "angle_plus_ring": available_features(df, GSG_ANGLE + GSG_RING),
        "all_features":    af,
    }
    return {k: v for k, v in sets.items() if len(v) >= 4}

def _GSG_delta_sets(df):
    meta = _GSG_META_DELTA
    all_d = [c for c in df.columns if c.startswith("delta__") and c not in meta]
    sets = {
        "delta_focus":  available_features(df, GSG_DELTA_FOCUS),
        "delta_radial": [c for c in all_d if ("rad_" in c or "dist_" in c)],
        "delta_angle":  [c for c in all_d if ("angle_" in c or "ring" in c or "imag_" in c or c.endswith("osc_frac"))],
        "delta_all":    all_d,
    }
    return {k: v for k, v in sets.items() if len(v) >= 4}

# ── Runners global spectral geometry ─────────────────────────────────────────────────────────────────
def _run_blocks(df, groupby_cols, tag_fn, fs_builder):
    """Runner genérico. tag_fn(keys) -> str."""
    if df.empty: return pd.DataFrame()
    feature_sets = fs_builder(df)
    results = []
    for keys, d in df.groupby(groupby_cols, dropna=False):
        if len(d) < MIN_ROWS_PER_BLOCK: continue
        if d["id"].nunique() < MIN_SUBJECTS_PER_BLOCK: continue
        tag = tag_fn(keys)
        for fn, fc in feature_sets.items():
            out = evaluate_block(d, fc, tag, fn)
            if not out.empty: results.append(out)
    return pd.concat(results, ignore_index=True) if results else pd.DataFrame()

def run_gsg_bywin(df):
    return _run_blocks(df, ["cond","win_sec","tercile"],
        lambda k: f"GSG_bywin__{k[0]}__{k[1]}s__{k[2]}",
        lambda d: _GSG_static_sets(d, _GSG_META_WIN))

def run_gsg_pooled(df):
    return _run_blocks(df, ["cond","tercile"],
        lambda k: f"GSG_pooled__{k[0]}__{k[1]}",
        lambda d: _GSG_static_sets(d, _GSG_META_POOL))

def run_gsg_delta_bywin(df):
    return _run_blocks(df, ["win_sec","tercile"],
        lambda k: f"GSG_delta_bywin__{k[0]}s__{k[1]}",
        lambda d: _GSG_delta_sets(d))

def run_GSG_delta_pooled(df):
    """Nuevo scope: delta pooled (si existe el CSV)."""
    return _run_blocks(df, ["tercile"],
        lambda k: f"GSG_delta_pooled__{k}",
        lambda d: _GSG_delta_sets(d))

print("✅ Bloque A (global spectral geometry) listo")

In [ ]:
# ▶️ EJECUTAR global spectral geometry ML
gsg_results = {}

for scope, runner, data_key in [
    ("bywin",        run_gsg_bywin,         "GSG_bywin"),
    ("pooled",       run_gsg_pooled,        "GSG_pooled"),
    ("delta_bywin",  run_gsg_delta_bywin,   "GSG_delta_bywin"),
    ("delta_pooled", run_GSG_delta_pooled,  "GSG_delta_pooled"),
]:
    print(f"Corriendo global spectral geometry {scope}...")
    r = runner(DATA[data_key])
    gsg_results[scope] = r
    if not r.empty:
        save_ranked_csv(r, OUT_GSG_RAW/f"GSG_ml_{scope}.csv")
        save_summary(r, OUT_GSG_SUMMARY, f"gsg_{scope}")
        print(f"  → {len(r)} filas guardadas")
    else:
        print("  → sin resultados")

# Ranking global global spectral geometry
GSG_all_list = [r.assign(scope=sc) for sc, r in gsg_results.items() if not r.empty]
if GSG_all_list:
    df_GSG_all = (pd.concat(GSG_all_list, ignore_index=True)
                   .sort_values(["subj_f1_macro_mean","subj_bal_acc_mean","subj_auc_mean","f1_macro_mean","auc_mean"],
                                ascending=[False]*5))
    save_ranked_csv(df_GSG_all, OUT_GSG_RANK/"GSG_ml_all_ranked_by_acc.csv")
    print("\nTOP 15 global spectral geometry:")
    cols = ["scope","tag","feature_set","model",
            "subj_f1_macro_mean","subj_bal_acc_mean","subj_auc_mean",
            "f1_macro_mean","auc_mean","threshold_mean","n_features","n_subjects"]
    print(df_GSG_all[[c for c in cols if c in df_GSG_all.columns]]
          .head(15).to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\n✅ global spectral geometry DONE")

---
# 🟣 BLOQUE B — dominant-mode spatial distribution ML

Sin cambios conceptuales respecto a la versión anterior, pero ahora comparte el mismo `evaluate_block` que global spectral geometry
(mismos modelos, misma limpieza, mismo threshold tuning).

In [ ]:
# ── Feature set builders dominant-mode spatial distribution ────────────────────────────────────────────────────
_DMSD_META_WIN  = {"pipeline_variant","id","group","y","cond","win_sec","tercile","kind",
                  "W_used","N_modes_total","participation_power","occipital_present_flag","npz_path"}
_DMSD_META_POOL = {"pipeline_variant","id","group","y","cond","win_sec","tercile","kind","n_win_pooled"}
_DMSD_META_DELTA = {"pipeline_variant","id","group","y","cond","win_sec","tercile","kind"}

def _DMSD_static_sets(df, meta):
    af = all_numeric_feature_cols(df, meta)
    sets = {
        "priority_stat":         available_features(df, DMSD_PRIORITY_STAT),
        "priority_no_occ":       available_features(df, DMSD_PRIORITY_NO_OCCIPITAL),
        "primary":               available_features(df, DMSD_PRIMARY),
        "distribution":          available_features(df, DMSD_DISTRIBUTION),
        "concentration":         available_features(df, DMSD_CONCENTRATION),
        "core_vs_others":        available_features(df, DMSD_CORE_VS_OTHERS),
        "disjoint":              available_features(df, DMSD_DISJOINT),
        "region_logits":         available_features(df, DMSD_REGION_LOGITS),
        "region_stars":          available_features(df, DMSD_REGION_STARS),
        "interpretable_compact": available_features(df, DMSD_INTERPRETABLE_COMPACT),
        "primary_plus_disjoint": available_features(df, DMSD_PRIMARY + DMSD_DISJOINT),
        "priority_combined":     available_features(df, DMSD_PRIORITY_STAT + DMSD_DISTRIBUTION),
        "interpretable_full":    available_features(df, DMSD_PRIMARY+DMSD_DISTRIBUTION+DMSD_DISJOINT+DMSD_REGION_LOGITS+DMSD_REGION_STARS),
        "all_features":          af,
    }
    return {k: dedupe_keep_order(v) for k, v in sets.items() if len(dedupe_keep_order(v)) >= 4}

def _DMSD_delta_sets(df):
    meta = _DMSD_META_DELTA
    af = dedupe_keep_order([c for c in df.columns if c.startswith("delta_") and c not in meta])
    sets = {
        "delta_priority":            available_features(df, DMSD_PRIORITY_DELTA),
        "delta_priority_plus_reorg": available_features(df, DMSD_PRIORITY_DELTA + DMSD_DELTA_REORG),
        "delta_reorg":               available_features(df, DMSD_DELTA_REORG),
        "delta_disjoint":            available_features(df, DMSD_DELTA_DISJOINT),
        "delta_compact":             available_features(df, DMSD_DELTA_COMPACT),
        "delta_all":                 af,
    }
    return {k: dedupe_keep_order(v) for k, v in sets.items() if len(dedupe_keep_order(v)) >= 4}

# ── Runners dominant-mode spatial distribution ─────────────────────────────────────────────────────────────────
def run_dmsd_bywin(df):
    return _run_blocks(df, ["pipeline_variant","cond","win_sec","tercile","kind"],
        lambda k: f"DMSD_bywin__{k[0]}__{k[1]}__{k[2]}s__{k[3]}__{k[4]}",
        lambda d: _DMSD_static_sets(d, _DMSD_META_WIN))

def run_dmsd_pooled(df):
    return _run_blocks(df, ["pipeline_variant","cond","tercile","kind"],
        lambda k: f"DMSD_pooled__{k[0]}__{k[1]}__{k[2]}__{k[3]}",
        lambda d: _DMSD_static_sets(d, _DMSD_META_POOL))

def run_dmsd_delta_bywin(df):
    return _run_blocks(df, ["pipeline_variant","win_sec","tercile","kind"],
        lambda k: f"DMSD_delta_bywin__{k[0]}__{k[1]}s__{k[2]}__{k[3]}",
        lambda d: _DMSD_delta_sets(d))

def run_DMSD_delta_pooled(df):
    return _run_blocks(df, ["pipeline_variant","tercile","kind"],
        lambda k: f"DMSD_delta_pooled__{k[0]}__{k[1]}__{k[2]}",
        lambda d: _DMSD_delta_sets(d))

print("✅ Bloque B (dominant-mode spatial distribution) listo")

In [ ]:
# ▶️ EJECUTAR dominant-mode spatial distribution ML
dmsd_results = {}

for scope, runner, data_key in [
    ("bywin",        run_dmsd_bywin,        "DMSD_bywin"),
    ("pooled",       run_dmsd_pooled,       "DMSD_pooled"),
    ("delta_bywin",  run_dmsd_delta_bywin,  "DMSD_delta_bywin"),
    ("delta_pooled", run_DMSD_delta_pooled, "DMSD_delta_pooled"),
]:
    print(f"Corriendo dominant-mode spatial distribution {scope}...")
    r = runner(DATA[data_key])
    dmsd_results[scope] = r
    if not r.empty:
        save_ranked_csv(r, OUT_DMSD_RAW/f"DMSD_ml_{scope}.csv")
        save_summary(r, OUT_DMSD_SUMMARY, f"dmsd_{scope}")
        print(f"  → {len(r)} filas guardadas")
    else:
        print("  → sin resultados")

# Ranking global dominant-mode spatial distribution
DMSD_all_list = [r.assign(scope=sc) for sc, r in dmsd_results.items() if not r.empty]
if DMSD_all_list:
    df_DMSD_all = (pd.concat(DMSD_all_list, ignore_index=True)
                   .sort_values(["subj_f1_macro_mean","subj_bal_acc_mean","subj_auc_mean","f1_macro_mean","auc_mean"],
                                ascending=[False]*5))
    save_ranked_csv(df_DMSD_all, OUT_DMSD_RANK/"DMSD_ml_all_ranked_by_acc.csv")
    print("\nTOP 20 dominant-mode spatial distribution:")
    cols = ["scope","tag","feature_set","model",
            "subj_f1_macro_mean","subj_bal_acc_mean","subj_auc_mean",
            "f1_macro_mean","auc_mean","threshold_mean","n_features","n_subjects"]
    print(df_DMSD_all[[c for c in cols if c in df_DMSD_all.columns]]
          .head(20).to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\n✅ dominant-mode spatial distribution DONE")

---
# 📊 BLOQUE C — Visualización

Lee los rankings desde disco. Puedes correr este bloque sin haber corrido A y B en esta sesión.

In [ ]:
def _load_if_exists(path):
    return pd.read_csv(path) if Path(path).exists() else pd.DataFrame()

df_gsg_ranked = _load_if_exists(OUT_GSG_RANK / "GSG_ml_all_ranked_by_acc.csv")
df_dmsd_ranked = _load_if_exists(OUT_DMSD_RANK / "DMSD_ml_all_ranked_by_acc.csv")

print(f"global spectral geometry ranked: {len(df_gsg_ranked)} filas")
print(f"dominant-mode spatial distribution ranked: {len(df_dmsd_ranked)} filas")

frames = []
if not df_gsg_ranked.empty: frames.append(df_gsg_ranked.assign(source="global spectral geometry"))
if not df_dmsd_ranked.empty: frames.append(df_dmsd_ranked.assign(source="dominant-mode spatial distribution"))

if not frames:
    print("⚠️  Sin datos — corre los bloques A y B primero.")
else:
    df_plot = pd.concat(frames, ignore_index=True).dropna(subset=["subj_f1_macro_mean"])
    print(f"Filas para visualización: {len(df_plot)}")

if frames:
    df_global_ranked = pd.concat(frames, ignore_index=True)
    save_ranked_csv(df_global_ranked, OUT_GLOBAL_RANK / "GSG_DMSD_global_ranked_by_acc.csv")
    print(f"CSV global guardado en: {OUT_GLOBAL_RANK / 'GSG_DMSD_global_ranked_by_acc.csv'}")

In [ ]:
# ── Plot 1: Distribución F1 — global spectral geometry vs dominant-mode spatial distribution y por scope ───────────────────────────
if frames:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    palette = {"global spectral geometry": "#4878CF", "dominant-mode spatial distribution": "#9B59B6"}

    sns.violinplot(data=df_plot, x="source", y="subj_f1_macro_mean",
                   palette=palette, inner="box", ax=axes[0])
    axes[0].axhline(0.5, color="gray", linestyle="--", linewidth=1)
    axes[0].set_title("subj F1-macro — global spectral geometry vs dominant-mode spatial distribution", fontweight="bold")
    axes[0].set_xlabel(""); axes[0].set_ylabel("F1-macro (sujeto)")

    scope_order = [s for s in ["bywin","pooled","delta_bywin","delta_pooled"] if s in df_plot["scope"].unique()]
    sns.violinplot(data=df_plot, x="scope", y="subj_f1_macro_mean",
                   hue="source", order=scope_order, palette=palette, inner="box", ax=axes[1])
    axes[1].axhline(0.5, color="gray", linestyle="--", linewidth=1)
    axes[1].set_title("subj F1-macro por scope", fontweight="bold")
    axes[1].set_xlabel(""); axes[1].set_ylabel(""); axes[1].legend(title="")

    plt.tight_layout()
    plt.savefig(OUT_GSG.parent / "viz_f1_distribution.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# ── Plot 2: Top modelos ────────────────────────────────────────────────────────
if frames:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    for ax, (src, color) in zip(axes, [("global spectral geometry","#4878CF"),("dominant-mode spatial distribution","#9B59B6")]):
        sub = df_plot[df_plot["source"]==src]
        if sub.empty: continue
        top = sub.groupby("model")["subj_f1_macro_mean"].mean().sort_values(ascending=False).head(14)
        top.plot(kind="barh", ax=ax, color=color, edgecolor="white")
        ax.axvline(0.5, color="gray", linestyle="--", linewidth=1)
        ax.set_title(f"{src} — mean subj F1 por modelo", fontweight="bold")
        ax.set_xlabel("mean subj F1-macro"); ax.set_ylabel(""); ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig(OUT_GSG.parent / "viz_top_models.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# ── Plot 3: Top feature sets ───────────────────────────────────────────────────
if frames:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    for ax, (src, color) in zip(axes, [("global spectral geometry","#4878CF"),("dominant-mode spatial distribution","#9B59B6")]):
        sub = df_plot[df_plot["source"]==src]
        if sub.empty: continue
        top = sub.groupby("feature_set")["subj_f1_macro_mean"].mean().sort_values(ascending=False).head(10)
        top.plot(kind="barh", ax=ax, color=color, edgecolor="white")
        ax.axvline(0.5, color="gray", linestyle="--", linewidth=1)
        ax.set_title(f"{src} — mean subj F1 por feature set", fontweight="bold")
        ax.set_xlabel("mean subj F1-macro"); ax.set_ylabel(""); ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig(OUT_GSG.parent / "viz_top_feature_sets.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# ── Plot 4: F1 vs AUC (top 40 global) ─────────────────────────────────────────
if frames:
    top40 = (df_plot.dropna(subset=["subj_f1_macro_mean","subj_auc_mean"])
                    .sort_values("subj_f1_macro_mean", ascending=False).head(40))
    fig, ax = plt.subplots(figsize=(9, 6))
    for src, grp in top40.groupby("source"):
        ax.scatter(grp["subj_auc_mean"], grp["subj_f1_macro_mean"],
                   label=src, color={"global spectral geometry":"#4878CF","dominant-mode spatial distribution":"#9B59B6"}[src],
                   alpha=0.8, s=60, edgecolors="white")
    ax.axhline(0.5, color="gray", linestyle="--", linewidth=1)
    ax.axvline(0.5, color="gray", linestyle="--", linewidth=1)
    ax.set_xlabel("subj AUC"); ax.set_ylabel("subj F1-macro")
    ax.set_title("Top 40 bloques — F1 vs AUC (sujeto)", fontweight="bold")
    ax.legend(title="Fuente")
    plt.tight_layout()
    plt.savefig(OUT_GSG.parent / "viz_f1_vs_auc_top40.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# ── Plot 5: Heatmap threshold medio por scope x modelo (top 8 modelos) ─────────
if frames:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    for ax, (src, cmap) in zip(axes, [("global spectral geometry","Blues"),("dominant-mode spatial distribution","Purples")]):
        sub = df_plot[(df_plot["source"]==src) & df_plot["threshold_mean"].notna()]
        if sub.empty: continue
        top_models = sub.groupby("model")["subj_f1_macro_mean"].mean().nlargest(8).index
        pivot = (sub[sub["model"].isin(top_models)]
                   .groupby(["scope","model"])["threshold_mean"]
                   .mean().unstack(fill_value=0.5))
        sns.heatmap(pivot, ax=ax, cmap=cmap, vmin=0.3, vmax=0.7,
                    annot=True, fmt=".2f", linewidths=0.4, cbar_kws={"shrink":0.7})
        ax.set_title(f"{src} — threshold medio por scope x modelo", fontweight="bold")
        ax.set_xlabel(""); ax.set_ylabel("")
    plt.tight_layout()
    plt.savefig(OUT_GSG.parent / "viz_threshold_heatmap.png", dpi=150, bbox_inches="tight")
    plt.show()